# 01b · 상태의 분류: 소통 클래스, 재귀와 일시, 주기 (Classification of States: Communicating Classes, Recurrence, Transience, Period)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.3 (닫힌 집합·기약·재귀/일시 정의, 7-상태 예제, 유한 닫힌 클래스 = 재귀) · Ross 12e §4.3 ($\sum_n p^n(i,i)=\infty$ 기준, 대칭 랜덤 워크) · Lawler 2e §1.3 (정준형) · Norris §1.2, §1.5 (재귀성은 클래스 성질) |
| 선수 노트북 | 01a |
| 예상 소요 | 90분 |
| 상태 | draft |

01a에서 도박꾼 파산 사슬은 시작점마다 $P^n$의 극한이 달랐고, 날씨 사슬은 하나의 극한으로 수렴했다. 그 차이를 설명하는 언어가 이 노트북이다: 상태공간을 **어디서 어디로 갈 수 있나**의 그래프로 보고, 그래프를 소통 클래스로 쪼갠 뒤, 각 클래스가 **새는가(일시)·갇히는가(재귀)**, 그리고 **몇 걸음 간격으로만 돌아올 수 있는가(주기)**를 묻는다.

## 0. 준비 (Setup)

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import matrix_power
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import (is_stochastic, simulate_chain, communicating_classes,
                          classify_states, absorption, hitting_times)
from spkit.plots import plot_hist_vs_pmf

SEED, rng = rng_for("01b")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_STEPS = 100             # 시간 지평 (일시 집합 탈출 확률은 4.3에서 행렬로 확인)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (01a) 마르코프 성질을 식으로 쓰고, '시간 동질(time-homogeneous)'이 무엇을 추가로 요구하는지 말하시오.

<details><summary>정답</summary>

$P(X_{n+1}=y\mid X_n=x,X_{n-1},\dots,X_0)=P(X_{n+1}=y\mid X_n=x)$ — 미래는 현재에만 의존한다. 시간 동질: 이 값이 $n$과 무관한 $p(x,y)$ 하나로 주어진다.

</details>

**Q2.** (01a) 채프먼-콜모고로프 방정식을 쓰고, 그것이 $P^{(n)}=P^n$을 뜻하는 이유를 한 줄로.

<details><summary>정답</summary>

$p^{m+n}(x,y)=\sum_z p^m(x,z)\,p^n(z,y)$. 우변은 행렬 곱 $(P^{(m)}P^{(n)})_{xy}$이므로 $P^{(m+n)}=P^{(m)}P^{(n)}$이고, $P^{(1)}=P$에서 귀납적으로 $P^{(n)}=P^n$.

</details>

**Q3.** (01a) 도박꾼 파산 사슬에서 $P^{300}$의 5번째 행은 어떤 모양이었나? 날씨 사슬의 $P^{10}$은?

<details><summary>정답</summary>

파산 사슬: $(0.7317,\,0,\dots,0,\,0.2683)$ — 흡수 상태 $0$과 $N$에만 질량이 남고 중간은 전부 $0$. 날씨 사슬: 두 행이 모두 $(0.5714,\,0.4286)$로 같아졌다(시작점을 잊는다).

</details>

**Q4.** (00c) 확률생성함수 $\varphi(s)=E[s^X]$에서 $E[X]$와 $P(X=0)$을 어떻게 읽나?

<details><summary>정답</summary>

$E[X]=\varphi'(1)$, $P(X=0)=\varphi(0)$.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **도달 가능(accessible)** $x\to y$: 어떤 $n\ge0$에 대해 $p^n(x,y)>0$. $n=0$을 허용하므로 항상 $x\to x$. **소통(communicate)** $x\leftrightarrow y$: $x\to y$이고 $y\to x$. 소통은 동치관계(반사·대칭·추이 — 추이성은 채프먼-콜모고로프에서 $p^{m+n}(x,z)\ge p^m(x,y)p^n(y,z)>0$)이고, 그 동치류가 **소통 클래스(communicating class)**다. 코드로는 유향 그래프 $\{(x,y):p(x,y)>0\}$의 **강연결성분(SCC)**이 곧 소통 클래스다 — `communicating_classes(P)`.
- **닫힌 집합(closed set)** $C$: $x\in C$, $p(x,y)>0\Rightarrow y\in C$ — 밖으로 나가는 화살표가 하나도 없다. **기약(irreducible)**: 상태공간 전체가 하나의 소통 클래스.
- **귀환시간(return time)** $T_x=\min\{n\ge1:X_n=x\}$ ($n\ge1$부터 센다), **귀환확률** $\rho_{xx}=P_x(T_x<\infty)$. 일반적으로 $\rho_{xy}=P_x(T_y<\infty)$, $T_y=\min\{n\ge1:X_n=y\}$ ($x$에서 출발해 $y$에 언젠가 도달할 확률); $\rho_{xx}$는 그 특별한 경우다. **재귀(recurrent)**: $\rho_{xx}=1$. **일시(transient)**: $\rho_{xx}<1$.
- **방문 횟수(number of visits)** $V_x=\sum_{n\ge0}\mathbf 1\{X_n=x\}$ (출발 포함). 강마르코프 성질에 의해 $x$에서 출발하면 $V_x\sim\text{Geometric}$: $P_x(V_x=k)=(1-\rho_{xx})\rho_{xx}^{k-1}$, $E_xV_x=\dfrac1{1-\rho_{xx}}=\sum_{n\ge0}p^n(x,x)$ (정리 1).
- **주기(period)** $d(x)=\gcd\{n\ge1:p^n(x,x)>0\}$. $d(x)=1$이면 **비주기(aperiodic)**. 주기는 클래스 성질(정리 4)이고, 자기 루프 $p(x,x)>0$가 하나라도 있으면 $1\in\{n:p^n(x,x)>0\}$이므로 그 클래스는 비주기다.
- **정준형(canonical form)**: 상태를 (일시 상태들, 닫힌 클래스들) 순으로 재배열하면
  $$P=\begin{pmatrix}Q&R\\0&P_{\text{closed}}\end{pmatrix}$$
  — $Q$는 일시 상태끼리, $R$은 일시 상태에서 닫힌 클래스로 새는 화살표, 왼쪽 아래 $0$은 '닫힘'. 01c의 출발점이다.

### 2.2 정리 1 — 기대 방문 횟수와 재귀 기준 (Ross 12e §4.3 Prop. 4.1, Durrett 3e §1.3)

> $x$에서 출발할 때 $V_x$는 성공확률 $1-\rho_{xx}$의 기하분포를 따르고
> $$E_xV_x=\sum_{n\ge0}p^n(x,x)=\frac1{1-\rho_{xx}}\quad(\rho_{xx}=1\text{이면 }\infty).$$
> 따라서 $x$가 재귀 $\iff\sum_n p^n(x,x)=\infty$.

**가정이 왜 필요한가**
- 강마르코프 성질(정지시간 $T_x$에서 다시 시작해도 같은 사슬): $x$에 돌아올 때마다 '또 돌아올 확률'이 똑같이 $\rho_{xx}$여야 시행이 독립 동일하게 반복되어 기하분포가 나온다.
- 시간 동질: 각 귀환 이후의 법칙이 같아야 한다. 전이확률이 시간에 따라 바뀌면 $k$번째 귀환의 확률이 $k$마다 달라진다.

<details><summary>증명</summary>

$k$번째 귀환 시각을 $T^{(1)}=T_x$, $T^{(k+1)}=\min\{n>T^{(k)}:X_n=x\}$로 두자 (앞의 것이 $\infty$이면 뒤도 $\infty$).

**(i) $P_x(T^{(k)}<\infty)=\rho_{xx}^k$.** $k=1$은 정의. $T^{(k)}<\infty$이면 그 시각의 상태는 $x$이고, 강마르코프 성질에 의해 $T^{(k)}$ 이후의 사슬은 $x$에서 새로 출발한 사슬과 같은 법칙을 따르며 $T^{(k)}$까지의 과거와 독립이다. 그러므로
$$P_x(T^{(k+1)}<\infty)=P_x(T^{(k)}<\infty)\cdot P_x(T_x<\infty)=\rho_{xx}^k\cdot\rho_{xx}.$$

**(ii) 기하분포.** $V_x\ge k+1\iff$ 출발 이후 적어도 $k$번 돌아옴 $\iff T^{(k)}<\infty$. 따라서 $P_x(V_x\ge k+1)=\rho_{xx}^k$ ($k\ge0$)이고, $P_x(V_x=k)=\rho_{xx}^{k-1}-\rho_{xx}^k=(1-\rho_{xx})\rho_{xx}^{k-1}$.

**(iii) 기댓값.** 비음 정수 확률변수의 꼬리합 공식으로 $E_xV_x=\sum_{k\ge0}P_x(V_x>k)=\sum_{k\ge0}\rho_{xx}^k$, 이것은 $\rho_{xx}<1$이면 $1/(1-\rho_{xx})$, $\rho_{xx}=1$이면 $\infty$.

**(iv) $\sum_np^n(x,x)$와의 동일성.** $V_x=\sum_{n\ge0}\mathbf 1\{X_n=x\}$이고 각 항이 비음이므로 단조수렴(Tonelli)으로 기댓값과 합을 바꿀 수 있다: $E_xV_x=\sum_{n\ge0}P_x(X_n=x)=\sum_{n\ge0}p^n(x,x)$.

(ii)–(iv)를 합치면 $\rho_{xx}=1\iff E_xV_x=\infty\iff\sum_np^n(x,x)=\infty$. $\square$

</details>

### 2.3 정리 2 — 재귀성은 클래스 성질 (Norris §1.5 Thm 1.5.3, Durrett 3e §1.3)

> $x$가 재귀이고 $x\leftrightarrow y$이면 $y$도 재귀다. 또한 $x$가 재귀이고 $x\to y$이면 $\rho_{yx}=1$이고 (따라서) $x\leftrightarrow y$.

**가정이 왜 필요한가**
- $x\to y$와 $y\to x$ 둘 다: $p^a(x,y)>0$, $p^b(y,x)>0$인 $a,b$가 있어야 $y$의 귀환 경로 안에 $x$의 귀환 경로를 끼워 넣어 $p^{a+n+b}(y,y)\ge p^b(y,x)\,p^n(x,x)\,p^a(x,y)$로 발산을 옮길 수 있다. 한쪽만 있으면 이 샌드위치가 만들어지지 않는다(일시 상태 $\to$ 재귀 상태는 흔하다).

<details><summary>증명</summary>

**첫 번째 주장.** $x\leftrightarrow y$이므로 $p^a(x,y)>0$, $p^b(y,x)>0$인 $a,b\ge0$이 있다. 채프먼-콜모고로프에서 임의의 $n\ge0$에 대해
$$p^{b+n+a}(y,y)\ \ge\ p^b(y,x)\,p^n(x,x)\,p^a(x,y).$$
양변을 $n$에 대해 합하면
$$\sum_{m\ge0}p^m(y,y)\ \ge\ \sum_{n\ge0}p^{b+n+a}(y,y)\ \ge\ p^b(y,x)\,p^a(x,y)\sum_{n\ge0}p^n(x,x)=\infty,$$
마지막 등호는 $x$가 재귀라는 정리 1의 기준. 다시 정리 1로 $y$는 재귀다.

**두 번째 주장.** $x\to y$이고 $\rho_{yx}<1$이라 하자. $x\to y$이므로 $x$에서 출발해 **$x$로 돌아오기 전에** $y$에 도달하는 사건 $A$의 확률은 양수다 ($p^a(x,y)>0$인 최단 경로 $x=z_0,z_1,\dots,z_a=y$에서 $x$가 중간에 다시 나오면 그 뒤만 잘라 쓰면 되므로, $x$를 거치지 않는 양의 확률 경로가 있다). $A$가 일어난 뒤 강마르코프 성질로 $y$에서 새로 출발한 사슬이 $x$에 영영 돌아오지 않을 확률은 $1-\rho_{yx}>0$. 두 사건이 이어지면 $T_x=\infty$이므로
$$1-\rho_{xx}\ \ge\ P_x(A)\,(1-\rho_{yx})>0,$$
즉 $\rho_{xx}<1$이 되어 $x$가 재귀라는 가정에 모순. 따라서 $\rho_{yx}=1$, 특히 $y\to x$이고 $x\leftrightarrow y$. $\square$

</details>

### 2.4 정리 3 — 유한 사슬의 분류 정리 (Durrett 3e §1.3 Lemma 1.5–Thm 1.7, Norris §1.5)

> 유한 상태공간에서
> (i) 닫힌 소통 클래스는 재귀, (ii) 닫히지 않은 클래스는 일시,
> (iii) 재귀 상태가 적어도 하나 존재하며, 일시 상태에서 출발하면 확률 1로 유한 시간 안에 어떤 닫힌 클래스로 흡수된다.

**가정이 왜 필요한가**
- **유한성**: 무한 사슬에서는 닫힌 기약 사슬도 일시적일 수 있다 — 편향 랜덤 워크는 $+\infty$로 달아나고, 3차원 대칭 랜덤 워크도 일시적이다(01x). 유한이면 '무한히 많은 시각을 유한 개의 상태가 나눠 가져야' 하므로 어딘가에는 무한히 자주 돌아오고, 그곳이 재귀 상태다.
- **닫힘**: 클래스 밖으로 나갈 확률이 양수이면, 나간 뒤에는 (클래스 정의상) 돌아올 수 없으므로 $\rho_{xx}<1$.

**증명 스케치.**
(ii) 클래스 $C$가 닫히지 않았으면 $x\in C$, $y\notin C$, $p(x,y)>0$인 쌍이 있고, $y\to x$가 아니다(아니면 $y\in C$). $C$의 임의 상태 $z$에서 $x$로 가는 양의 확률 경로(도중에 $z$를 다시 지나지 않도록 잘라낸 것) 뒤에 $x\to y$를 붙이면, $z$에서 출발해 $z$로 돌아오지 못하는 사건의 확률이 양수이므로 $\rho_{zz}<1$.
(iii) 일시 상태 $z$는 정리 1로 $E_zV_z<\infty$ — 유한 번만 방문된다. 일시 상태가 유한 개이므로 어느 시각 이후로는 일시 상태를 전혀 방문하지 않고, 그 시점부터는 (다시 새지 않는) 닫힌 클래스 안에 있다. 재귀 상태가 없다면 모든 상태가 유한 번만 방문되어 무한히 많은 시각을 채울 수 없으므로 모순.
(i) 닫힌 클래스 $C$에서 출발하면 영원히 $C$ 안에 머물고, $C$가 유한이므로 어떤 $x\in C$가 무한히 자주 방문된다 — 그 $x$는 (강마르코프 논법으로) 재귀이고, 정리 2로 $C$ 전체가 재귀다. 완전한 증명은 Durrett 3e §1.3, Norris §1.5.

### 2.5 정리 4 — 주기는 클래스 성질

> $x\leftrightarrow y$이면 $d(x)=d(y)$.

**가정이 왜 필요한가**
- $p^a(x,y)p^b(y,x)>0$인 $a,b$가 있어야 $a+b$와 $a+n+b$가 **모두** $x$의 귀환 길이임을 쓸 수 있다. 두 귀환 길이의 차가 $n$이라는 것이 논증의 전부다.

<details><summary>증명</summary>

$p^a(x,y)>0$, $p^b(y,x)>0$인 $a,b\ge1$을 잡자 ($x\ne y$이면 자동으로 $\ge1$; $x=y$는 자명). 그러면 $p^{a+b}(x,x)\ge p^a(x,y)p^b(y,x)>0$이므로 $d(x)\mid a+b$.

이제 $p^n(y,y)>0$인 임의의 $n\ge1$을 잡으면 $p^{a+n+b}(x,x)\ge p^a(x,y)\,p^n(y,y)\,p^b(y,x)>0$이므로 $d(x)\mid a+n+b$. 두 배수의 차를 취하면 $d(x)\mid(a+n+b)-(a+b)=n$.

즉 $d(x)$는 $\{n\ge1:p^n(y,y)>0\}$의 모든 원소를 나누므로 그 최대공약수 $d(y)$도 나눈다: $d(x)\mid d(y)$. $x,y$의 역할을 바꾸면 $d(y)\mid d(x)$. 둘 다 양의 정수이므로 $d(x)=d(y)$. $\square$

</details>

### 2.6 직관

상태공간을 '어디서 어디로 갈 수 있나'의 그래프로 보면 강연결성분이 소통 클래스다. 화살표가 밖으로 **새는** 클래스는 (유한 사슬에서) 언젠가 반드시 떠나므로 일시적이고, 새지 않는 클래스는 그 안에 갇혀 무한히 돌아오므로 재귀적이다. 01a의 파산 사슬은 $\{0\},\{10\}$이 닫힌 클래스, $\{1,\dots,9\}$가 새는 클래스라서 시작점마다 극한이 달랐고, 날씨 사슬은 기약이라 극한이 하나였다.

주기는 **별개의 축**이다. 반사 랜덤 워크는 재귀적이지만 짝수 시각에만 돌아올 수 있어 $p^n(0,0)$이 $0$과 양수를 번갈아 간다 — 재귀인데도 $P^n$이 수렴하지 않는다. 01d의 수렴 정리가 '기약 + 비주기'를 둘 다 요구하는 이유다.

### 2.7 함정(traps)

1. 주기 $d=3$이라고 정확히 3걸음마다 돌아온다는 뜻이 아니다. gcd이므로 귀환 길이가 $\{6,9,15,\dots\}$처럼 3의 배수 중 일부만 가능할 수도 있다.
2. '재귀'는 유한 사슬에서는 '닫힌 클래스 소속'과 같지만, 무한 사슬에서는 닫혀 있어도 일시적일 수 있다(01x의 3차원 걷기).
3. `communicating_classes`가 돌려주는 클래스의 **순서(index)는 구현 의존적**이다. 클래스 index가 아니라 상태 집합(`set`/`frozenset`)으로 비교할 것.
4. 귀환시간 $T_x$는 $n\ge1$부터 센다. `hitting_times(paths, x)`를 그대로 쓰면 출발 상태 때문에 $0$이 나온다. `paths[:, 1:]`에 적용하고 1을 더할 것.
5. 일시 상태의 귀환확률을 시뮬레이션할 때 시간 지평 $n$을 짧게 잡으면 $\rho$를 과소추정한다(아직 돌아오지 않은 경로를 '못 돌아온' 것으로 센다). 지평 이후에도 일시 집합에 남아 있을 확률이 무시할 만한지($<10^{-6}$) $Q^n$의 행합으로 확인할 것.

✍️ **내 말로**: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요) — (1) 재귀/일시의 정의, (2) '유한 닫힌 클래스는 재귀'의 이유, (3) 주기가 왜 '돌아오는 간격'이 아니라 gcd로 정의되는지를 각각 두 문장으로.

## 3. Predict — 코드를 돌리기 전에 예측

아래 7-상태 예제 사슬(상태 $0,\dots,6$)에 대한 질문입니다. 행렬은 4절 첫 셀에 있습니다(행 $x$ = 상태 $x$에서의 전이확률):

$$
P=\begin{pmatrix}
0.5&0.5&0&0&0&0&0\\
0.3&0.7&0&0&0&0&0\\
0.2&0&0.3&0.5&0&0&0\\
0&0&0.4&0.2&0.4&0&0\\
0&0&0&0&0&1&0\\
0&0&0&0&0&0&1\\
0&0&0&0&1&0&0
\end{pmatrix}
$$

`None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향이 맞는지가 목적입니다. 예측을 먼저 적어 두고 4절을 실행하세요.

In [ ]:
predictions = {
    # 상태 0 의 귀환확률 rho_00 은? (0 이 속한 클래스는 닫혀 있나?)
    "rho_00": None,
    # 0 에서 출발할 때 평균 귀환시간 E_0[T_0] 은?
    # (힌트: 0 에 머물 확률 0.5, 1 로 간 뒤에는 매 걸음 0.3 의 확률로 돌아옴)
    "mean_return_0": None,
    # 일시 상태 2 의 귀환확률 rho_22 는? (직접 귀환 0.3 + 3 으로 갔다가 되돌아오는 경우)
    "rho_22": None,
    # 2 에서 출발할 때 상태 2 의 총 방문 횟수(출발 포함) 기댓값 E_2[V_2] 는?
    "mean_visits_2": None,
    # 2 에서 출발할 때 결국 순환 클래스 {4,5,6} 에 흡수될 확률은? (나머지는 {0,1} 로)
    "p_reach_cycle_from_2": None,
    # 클래스 {4,5,6} 의 주기는?
    "period_cycle_class": None,
    # 반사 랜덤 워크 {0,...,4} (양 끝에서 확률 1 로 안쪽, 안쪽은 1/2 씩) 의 주기는?
    "period_reflecting_walk": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 예제 사슬과 클래스 구조

먼저 행렬을 정의하고, 전이 그래프의 강연결성분으로 소통 클래스를 구한다. 기대: $\{0,1\}$(닫힘, 주기 1), $\{2,3\}$(일시, 주기 1), $\{4,5,6\}$(닫힘, 주기 3). 클래스 순서는 구현 의존이므로 `frozenset`으로 비교한다(함정 3).

In [ ]:
P = np.array([
    [0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0],   # 0: 닫힌 클래스 {0,1}
    [0.3, 0.7, 0.0, 0.0, 0.0, 0.0, 0.0],   # 1
    [0.2, 0.0, 0.3, 0.5, 0.0, 0.0, 0.0],   # 2: 일시 클래스 {2,3}, 0 으로 샘
    [0.0, 0.0, 0.4, 0.2, 0.4, 0.0, 0.0],   # 3: 4 로 샘
    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0],   # 4: 순환 4 -> 5 -> 6 -> 4
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0],   # 5
    [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0],   # 6
])
assert is_stochastic(P)
classes = communicating_classes(P)
info = classify_states(P)
print("classes           :", classes)
print("closed (recurrent):", [classes[i] for i in info["recurrent"]])
print("transient states  :", info["transient_states"])
print("period by class   :", {tuple(classes[i]): d for i, d in info["period"].items()})

assert {frozenset(c) for c in classes} == {frozenset({0, 1}), frozenset({2, 3}), frozenset({4, 5, 6})}
assert set(info["transient_states"]) == {2, 3} and set(info["recurrent_states"]) == {0, 1, 4, 5, 6}

상태를 정준형 순서 (일시 $2,3$ | 닫힌 $0,1$ | 닫힌 $4,5,6$)로 재배열해 $P>0$의 모양을 그리면 $\begin{pmatrix}Q&R\\0&P_{\text{closed}}\end{pmatrix}$ 구조가 눈에 보인다.

In [ ]:
order = [2, 3, 0, 1, 4, 5, 6]                     # canonical order: transient | closed | closed
A = (P[np.ix_(order, order)] > 0)
fig, ax = plt.subplots(figsize=(6.4, 5.4))
ax.imshow(A, cmap="Greys", vmin=0, vmax=1.6)
# (row0, col0, n_rows, n_cols, color, label) in the reordered matrix
blocks = [(0, 0, 2, 2, "C3", "Q"), (0, 2, 2, 5, "C1", "R"),
          (2, 2, 2, 2, "C0", "closed {0,1}"), (4, 4, 3, 3, "C0", "closed {4,5,6}")]
for r0, c0, nr, nc, color, label in blocks:
    ax.add_patch(plt.Rectangle((c0 - 0.5, r0 - 0.5), nc, nr, fill=False, lw=2.2, ec=color))
    ax.text(c0 + nc - 0.55, r0 - 0.3, label, color=color, fontsize=9, fontweight="bold", ha="right", va="top",
            bbox=dict(fc="white", ec="none", alpha=0.85, pad=1.5))
ax.set_xticks(range(7), [str(s) for s in order]); ax.set_yticks(range(7), [str(s) for s in order])
ax.set_xlabel("to state (canonical order)"); ax.set_ylabel("from state"); ax.grid(False)
ax.set_title("Transition graph structure:\ntransient block Q, leaks R, closed classes")
plt.show()

### 4.2 재귀 상태 0의 귀환시간

$\{0,1\}$은 닫힌 클래스이므로 정리 3에 의해 재귀: **모든** 경로가 0으로 돌아와야 한다. 귀환시간은 $n\ge1$부터 세므로 `paths[:, 1:]`에 `hitting_times`를 적용하고 1을 더한다(함정 4). 정확한 분포: $T_0=1$일 확률은 $0.5$ (0에 머묾)이고, 아니면 1로 간 뒤 매 걸음 $0.3$으로 돌아오므로 $P(T_0=k)=0.5\cdot0.3\cdot0.7^{k-2}$ ($k\ge2$).

In [ ]:
paths0 = simulate_chain(P, 0, N_STEPS, rng, n_paths=N_PATHS)
T0 = hitting_times(paths0[:, 1:], 0) + 1          # 귀환시간 (n >= 1 부터)
assert np.isfinite(T0).all()                       # 재귀: 모든 경로가 돌아온다 (rho_00 = 1)

est_rho00 = mc_proportion(np.isfinite(T0))
est_T0 = mc_estimate(T0)
print(f"rho_00 hat  = {est_rho00}")
print(f"E_0[T_0] hat = {est_T0}   (exact 8/3 = {8/3:.4f})")
print(f"max T_0 observed = {int(T0.max())}  (horizon {N_STEPS})")

In [ ]:
def return_time_pmf_state0(k):
    """helper (spkit에 없어서 여기서 정의): P_0(T_0 = k) for the example chain."""
    k = np.asarray(k, dtype=float)
    return np.where(k == 1, 0.5, 0.5 * 0.3 * 0.7 ** (k - 2))

ks = np.arange(1, 21)
pmf_T0 = return_time_pmf_state0(ks)
pmf_T0[-1] = 1 - pmf_T0[:-1].sum()                 # 마지막 점 = 꼬리 질량 P(T_0 >= 20) (막대와 맞추기 위해)
ax = plot_hist_vs_pmf(np.minimum(T0, 20), ks, pmf_T0, label_pmf="exact pmf (last point = tail mass)")
ax.set_xlabel("return time $T_0$ (values >= 20 pooled into the last bar)"); ax.set_ylabel("probability")
ax.set_title("Return time to recurrent state 0: mean 8/3")
plt.show()

### 4.3 일시 상태 2: 귀환확률, 방문 횟수, 흡수 방향

$\{2,3\}$은 $0$과 $4$로 새므로 일시적. 세 가지를 잰다: (a) 귀환 여부의 비율 → $\rho_{22}$, (b) 방문 횟수 $V_2$ → 기하분포, 평균 $1/(1-\rho_{22})$, (c) 지평 끝에서 $\{4,5,6\}$에 있는 비율 → 흡수확률. 먼저 지평 절단이 안전한지(함정 5) $Q^{n}$의 행합 = "$n$걸음 뒤에도 $\{2,3\}$ 안에 있을 확률"로 확인한다.

In [ ]:
Q = P[np.ix_([2, 3], [2, 3])]
p_still_transient = matrix_power(Q, N_STEPS).sum(axis=1)   # from 2, from 3
print(f"P(still in {{2,3}} after {N_STEPS} steps) = {p_still_transient}")
assert p_still_transient.max() < 1e-6

paths2 = simulate_chain(P, 2, N_STEPS, rng, n_paths=N_PATHS)
T2 = hitting_times(paths2[:, 1:], 2) + 1
V2 = (paths2 == 2).sum(axis=1)                        # 방문 횟수 (출발 포함)
in_cycle = np.isin(paths2[:, -1], [4, 5, 6])          # 지평 끝에서 순환 클래스에 있는가

est_rho22 = mc_proportion(np.isfinite(T2))
est_V2 = mc_estimate(V2)
est_cycle = mc_proportion(in_cycle)
print(f"rho_22 hat          = {est_rho22}")
print(f"E_2[V_2] hat        = {est_V2}")
print(f"P_2(absorbed in cycle) hat = {est_cycle}")
print(f"final states seen: {sorted(set(paths2[:, -1].tolist()))}  (2,3 should be absent)")

In [ ]:
N_fund = absorption(P, [2, 3], [0, 1, 4, 5, 6])["N"]   # N = (I - Q)^{-1}, 기대 방문 횟수 행렬
rho22_exact = 1 - 1 / N_fund[0, 0]                      # E_2 V_2 = 1/(1 - rho_22)
kv = np.arange(1, 13)
pmf_V2 = (1 - rho22_exact) * rho22_exact ** (kv - 1)
pmf_V2[-1] = 1 - pmf_V2[:-1].sum()                 # 마지막 점 = 꼬리 질량 P(V_2 >= 12) = rho^11
ax = plot_hist_vs_pmf(np.minimum(V2, 12), kv, pmf_V2, label_pmf="geometric pmf (1-rho) rho^(k-1); last = tail")
ax.set_xlabel("number of visits $V_2$ (values >= 12 pooled into the last bar)"); ax.set_ylabel("probability")
ax.set_title("Visits to transient state 2: geometric, mean 20/9")
plt.show()

### 4.4 주기: $p^n(x,x)$의 진동

(a) 순환 클래스 $\{4,5,6\}$: $p^n(4,4)$는 $n$이 3의 배수일 때만 양수. (b) 반사 랜덤 워크 $\{0,\dots,4\}$: 이분 그래프이므로 짝수 걸음에만 돌아오고, $p^n(0,0)$은 수렴하지 않고 $0$과 양수를 번갈아 간다 (짝수 부분열 $p^{2m}(0,0)$은 $1/4=2\pi(0)$로 수렴하지만 — $\pi\propto(1,2,2,2,1)$ — 전체 수열은 진동한다). (c) 게으른(lazy) 사슬 $P'=0.9P+0.1I$: 자기 루프 하나로 주기가 1이 되고 $p^n(4,4)\to1/3$.

In [ ]:
def reflecting_walk_matrix(N):
    """helper (spkit에 없어서 여기서 정의): reflecting SRW on {0..N}, p(0,1)=p(N,N-1)=1, 1/2 inside."""
    R = np.zeros((N + 1, N + 1))
    R[0, 1] = 1.0
    R[N, N - 1] = 1.0
    for i in range(1, N):
        R[i, i - 1] = R[i, i + 1] = 0.5
    return R

P_ref = reflecting_walk_matrix(4)
P_lazy = 0.9 * P + 0.1 * np.eye(7)
ns = np.arange(1, 13)
pn_cycle = np.array([matrix_power(P, n)[4, 4] for n in ns])
pn_ref = np.array([matrix_power(P_ref, n)[0, 0] for n in ns])
pn_lazy = np.array([matrix_power(P_lazy, n)[4, 4] for n in ns])
print("n            :", ns)
print("p^n(4,4)     :", np.round(pn_cycle, 3))
print("p^n(0,0) ref :", np.round(pn_ref, 4))
print("p^n(4,4) lazy:", np.round(pn_lazy, 3), f"... p^200(4,4) lazy = {matrix_power(P_lazy, 200)[4, 4]:.5f}")

`classify_states`의 주기(BFS 레벨의 gcd)가 위 패턴과 일치하는지 확인한다. 반사 걷기는 기약이라 클래스가 하나뿐이므로 index 0을 써도 되지만, 예제 사슬에서는 상태 집합으로 클래스를 찾는다(함정 3).

In [ ]:
period_cycle = info["period"][[i for i, c in enumerate(classes) if set(c) == {4, 5, 6}][0]]
period_ref = classify_states(P_ref)["period"][0]          # 반사 걷기는 기약: 클래스 하나
lazy_info = classify_states(P_lazy)
period_lazy = lazy_info["period"][[i for i, c in enumerate(lazy_info["classes"]) if set(c) == {4, 5, 6}][0]]
print(f"periods: cycle class {period_cycle}, reflecting walk {period_ref}, lazy cycle class {period_lazy}")
assert [matrix_power(P, n)[4, 4] > 0 for n in range(1, 13)] == [n % 3 == 0 for n in range(1, 13)]
assert period_cycle == 3 and period_ref == 2 and period_lazy == 1

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, vals, title in zip(axes, [pn_cycle, pn_ref, pn_lazy],
                           ["cycle class, state 4 (period 3)", "reflecting walk, state 0 (period 2)",
                            "lazy cycle, state 4 (period 1)"]):
    ax.stem(ns, vals, basefmt=" ")
    ax.set_xticks(ns); ax.set_xlabel("n"); ax.set_title(title, fontsize=10)
axes[1].axhline(1 / 4, color="C3", ls="--", lw=1, label="even-subsequence limit 1/4"); axes[1].legend()
axes[2].axhline(1 / 3, color="C3", ls="--", lw=1, label="limit 1/3"); axes[2].legend()
axes[0].set_ylabel("$p^n(x,x)$")
fig.suptitle("Periodicity: $p^n(x,x)$ vanishes off the multiples of d", y=1.02)
plt.tight_layout(); plt.show()

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드로 계산한다: $E_0T_0$은 pmf의 합, $\rho_{22}$·$E_2V_2$는 기본행렬 $N=(I-Q)^{-1}$, 흡수확률은 $B=NR$ (01c에서 다시 유도), $\rho_{00}$은 정리 3(닫힌 클래스 $\Rightarrow$ 재귀)을 `classify_states`의 재귀 판정으로, 주기는 정의 그대로 $\gcd\{n\le12:p^n(x,x)>0\}$ (`math.gcd`)로 계산하고 `classify_states`의 BFS 주기를 estimate 열에 둔다.

In [ ]:
kk = np.arange(1, 3000)
mean_return_0_exact = float((kk * return_time_pmf_state0(kk)).sum())     # = 8/3
rho00_exact = 1.0 if 0 in info["recurrent_states"] else float("nan")      # 닫힌 클래스 = 재귀 (정리 3)
mean_visits_2_exact = float(N_fund[0, 0])                                  # 20/9
absb = absorption(P, [2, 3], [0, 1, 4, 5, 6])["B"]                        # rows: from 2, 3; cols: 0,1,4,5,6
p_cycle_exact = float(absb[0, 2:].sum())                                  # 5/9
period_cycle_exact = math.gcd(*[n for n in range(1, 13) if matrix_power(P, n)[4, 4] > 0])       # 정의: gcd
period_ref_exact = math.gcd(*[n for n in range(1, 13) if matrix_power(P_ref, n)[0, 0] > 0])

rows = [
    {"name": "rho_00 (return prob., recurrent)", "predicted": predictions["rho_00"], "estimate": est_rho00, "exact": rho00_exact},
    {"name": "E_0[T_0] (mean return time)", "predicted": predictions["mean_return_0"], "estimate": est_T0, "exact": mean_return_0_exact},
    {"name": "rho_22 (return prob., transient)", "predicted": predictions["rho_22"], "estimate": est_rho22, "exact": rho22_exact},
    {"name": "E_2[V_2] (mean visits)", "predicted": predictions["mean_visits_2"], "estimate": est_V2, "exact": mean_visits_2_exact},
    {"name": "P_2(absorbed in {4,5,6})", "predicted": predictions["p_reach_cycle_from_2"], "estimate": est_cycle, "exact": p_cycle_exact},
    {"name": "period of class {4,5,6} (not MC)", "predicted": predictions["period_cycle_class"], "estimate": float(period_cycle), "exact": float(period_cycle_exact), "se": 0.0},
    {"name": "period of reflecting walk (not MC)", "predicted": predictions["period_reflecting_walk"], "estimate": float(period_ref), "exact": float(period_ref_exact), "se": 0.0},
]
print(f"exact: E_0T_0={mean_return_0_exact:.6f}, rho_22={rho22_exact:.6f}, E_2V_2={mean_visits_2_exact:.6f}, P_2(cycle)={p_cycle_exact:.6f}")
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **rho_00**: 1보다 작게 나오면 지평이 너무 짧거나(함정 5) $\{0,1\}$이 닫혀 있지 않은 것 — 정리 3의 (i)가 깨진다.
- **E_0[T_0]**: 어긋나면 `paths[:, 1:]`로 자르지 않아 출발 시각 0이 섞였거나(함정 4) pmf의 $k\ge2$ 항을 잘못 쓴 것.
- **rho_22**: 어긋나면 1단계 분석 $0.3+0.5\cdot0.5$ 또는 $1-1/N_{22}$ 중 하나가 틀린 것; 시뮬레이션 쪽이 작다면 지평 절단.
- **E_2[V_2]**: 어긋나면 방문 횟수가 기하분포가 아니라는 뜻 — 강마르코프 성질(정리 1의 가정)이 의심된다.
- **P_2(absorbed in {4,5,6})**: 어긋나면 $B=NR$의 열 순서(0,1 | 4,5,6)를 잘못 읽었거나 지평 끝에 아직 $\{2,3\}$에 남은 경로가 있는 것.
- **period 두 행**: MC가 아니므로 SE $=0$ (exact = gcd 정의, estimate = `classify_states`의 BFS 주기; 둘이 다르면 spkit 버그). 예측이 어긋났다면 '주기 = 돌아오는 간격'으로 오해한 것(함정 1).

In [ ]:
assert_close(est_rho00, rho00_exact, k=4, name="rho_00")
assert_close(est_T0, mean_return_0_exact, k=4, name="mean_return_0")
assert_close(est_rho22, rho22_exact, k=4, name="rho_22")
assert_close(est_V2, mean_visits_2_exact, k=4, name="mean_visits_2")
assert_close(est_cycle, p_cycle_exact, k=4, name="p_reach_cycle_from_2")
print("all checks passed")

## 6. 연습문제

### E1 계산

5-상태 사슬
$$P=\begin{pmatrix}0.5&0.5&0&0&0\\0.5&0.5&0&0&0\\0.25&0&0.5&0.25&0\\0&0&0&0&1\\0&0&0&1&0\end{pmatrix}$$
의 소통 클래스, 닫힘 여부, 재귀/일시, 각 클래스의 주기를 **손으로** 구하시오. 상태 2의 $\rho_{22}$와 $E_2V_2$도 구하시오. 다 푼 뒤 `classify_states`로 검산하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

그래프를 그리면 화살표는 $0\leftrightarrow1$ (자기 루프 포함), $2\to0$, $2\to2$, $2\to3$, $3\to4$, $4\to3$.

- $\{0,1\}$: 서로 소통, 밖으로 나가는 화살표 없음 → **닫힘, 재귀**. 자기 루프 $p(0,0)>0$ → **주기 1**.
- $\{2\}$: $2\to0$, $2\to3$으로 새고 돌아올 수 없음 → **닫히지 않음, 일시**. 자기 루프 → 주기 1.
- $\{3,4\}$: $3\to4\to3$만 가능 → **닫힘, 재귀**. 귀환 길이 $\{2,4,6,\dots\}$ → **주기 2**.

$\rho_{22}$: 2에서 한 걸음에 2로 머무는 것($0.5$) 외에는 $0$ 또는 $3$으로 가고, 둘 다 2로 돌아올 수 없으므로 $\rho_{22}=p(2,2)=0.5$. $E_2V_2=1/(1-0.5)=2$.

검산 코드:
```python
P5 = np.array([[0.5, 0.5, 0, 0, 0], [0.5, 0.5, 0, 0, 0], [0.25, 0, 0.5, 0.25, 0],
               [0, 0, 0, 0, 1], [0, 0, 0, 1, 0]])
info5 = classify_states(P5)
print(info5["classes"], info5["transient_states"], info5["period"])
# [[0, 1], [2], [3, 4]] [2] {0: 1, 1: 1, 2: 2}   (클래스 순서는 구현 의존 — 함정 3; 아래처럼 집합으로 비교)
assert {frozenset(c) for c in info5["classes"]} == {frozenset({0, 1}), frozenset({2}), frozenset({3, 4})}
i34 = [i for i, c in enumerate(info5["classes"]) if set(c) == {3, 4}][0]
assert info5["period"][i34] == 2 and set(info5["transient_states"]) == {2}
N5 = absorption(P5, [2], [0, 1, 3, 4])["N"]
print(N5[0, 0], 1 - 1 / N5[0, 0])          # 2.0  0.5
```

</details>

### E2 유도 후 시뮬레이션 검증

예제 7-상태 사슬에서 **1단계 분석(first-step analysis)**으로 $\rho_{22}$와 $\rho_{33}$을 유도하시오. 그런 다음 상태 3에서 출발하는 경로 `N_PATHS`개(지평 100)로 $\rho_{33}$을 추정해 4 SE 안에서 일치하는지 `assert_close`로 확인하시오. 검산: $N=(I-Q)^{-1}$의 대각에서 $\rho_{xx}=1-1/N_{xx}$.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$\{2,3\}$을 떠나면 돌아올 수 없으므로 '떠나기 전에 목표에 닿을 확률'만 세면 된다.

$a=P_3(\text{2에 닿음, } \{2,3\}\text{을 떠나기 전})$: 첫 걸음이 $2$($0.4$)면 성공, $3$에 머물면($0.2$) 처음부터, $4$($0.4$)면 실패. $a=0.4+0.2a\Rightarrow a=0.5$.

$b=P_2(\text{3에 닿음})$: $b=0.5+0.3b\Rightarrow b=5/7$.

$\rho_{22}=p(2,2)+p(2,3)\,a=0.3+0.5\cdot0.5=0.55$.
$\rho_{33}=p(3,3)+p(3,2)\,b=0.2+0.4\cdot\tfrac57=0.2+\tfrac27=\tfrac{17}{35}\approx0.485714$.

검산: $N=(I-Q)^{-1}=\begin{pmatrix}20/9&25/18\\10/9&35/18\end{pmatrix}$이므로 $\rho_{22}=1-9/20=0.55$, $\rho_{33}=1-18/35=17/35$.

시뮬레이션 코드 (SE $\approx0.0035$ at $n=20000$):
```python
paths3 = simulate_chain(P, 3, N_STEPS, rng, n_paths=N_PATHS)
T3 = hitting_times(paths3[:, 1:], 3) + 1
est_rho33 = mc_proportion(np.isfinite(T3))
rho33_exact = 1 - 1 / N_fund[1, 1]                 # = 17/35
print(est_rho33, rho33_exact, 17 / 35)
assert_close(est_rho33, rho33_exact, k=4, name="rho_33")
```

</details>

### E3 가정을 깨보기

순환 클래스 $\{4,5,6\}$은 주기 3이라 $p^n(4,4)$가 수렴하지 않는다.
(a) 각 상태에 자기 루프 $0.1$을 추가한 게으른 사슬 $P'=0.9P+0.1I$에서 주기와 $p^n(4,4)$의 거동이 어떻게 바뀌는지 관찰하시오 (4.4의 결과를 설명하시오).
(b) 자기 루프 대신 **$6\to5$ 화살표 하나**만 추가하면($p(6,5)=0.1$, $p(6,4)=0.9$) 주기는? 귀환 길이 집합을 써서 gcd로 설명하고 `classify_states`와 `matrix_power`로 확인하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 자기 루프가 있으면 $1\in\{n:p'^n(4,4)>0\}$이므로 **주기 1**(비주기). 재귀·기약(클래스 안에서)·비주기이므로 01d의 수렴 정리가 적용되어 $p'^n(4,4)\to\pi(4)$. 클래스 $\{4,5,6\}$ 위의 $P'$는 이중확률행렬(열합도 1)이라 정상분포는 균등: 극한 $1/3$. 4.4에서 $p'^{200}(4,4)=0.33333$이 이것이다.

(b) 길이 3 순환 $4\to5\to6\to4$와 길이 5 경로 $4\to5\to6\to5\to6\to4$가 모두 양의 확률. $\gcd(3,5)=1$이므로 **주기 1** — 자기 루프가 하나도 없는데도 비주기다. 주기는 '한 번에 돌아오는 간격'이 아니라 귀환 길이들의 gcd다(함정 1). $p^n(4,4)$는 $n=1,2,4$에서 $0$이지만 $n\ge5$부터는 $3$의 배수가 아니어도 양수가 되고, $n\to\infty$에서 정상분포 값으로 수렴한다.

확인 코드:
```python
P_b = P.copy()
P_b[6] = [0, 0, 0, 0, 0.9, 0.1, 0]                 # 6 -> 4 (0.9), 6 -> 5 (0.1)
assert is_stochastic(P_b)
info_b = classify_states(P_b)
cyc = [i for i, c in enumerate(info_b["classes"]) if set(c) == {4, 5, 6}][0]
print("period:", info_b["period"][cyc])            # 1
print([round(matrix_power(P_b, n)[4, 4], 4) for n in range(1, 13)])
# [0, 0, 0.9, 0, 0.09, 0.81, 0.009, 0.162, 0.7299, 0.0243, 0.2188, 0.6593]
print(matrix_power(P_b, 500)[4, 4])                 # pi(4) on the class, = 9/29 ~ 0.3103
```

</details>

## 7. 정리

1. 소통 클래스 = 전이 그래프의 강연결성분; 닫힌 클래스는 나가는 화살표가 없다. 정준형 $\begin{pmatrix}Q&R\\0&P_{\text{closed}}\end{pmatrix}$로 재배열하면 구조가 한눈에 보인다.
2. 재귀 $\iff\rho_{xx}=1\iff\sum_np^n(x,x)=\infty$; 일시 상태의 방문 횟수는 기하분포이고 평균은 $1/(1-\rho_{xx})=N_{xx}$.
3. 유한 사슬: 닫힌 클래스 = 재귀, 새는 클래스 = 일시; 일시 상태에서 출발하면 확률 1로 닫힌 클래스에 흡수된다. '유한'을 빼면 닫혀 있어도 일시적일 수 있다.
4. 주기 $d(x)=\gcd\{n:p^n(x,x)>0\}$는 클래스 성질; 자기 루프 하나로 비주기가 된다. 주기적이면 $p^n(x,x)$는 수렴하지 않고 진동한다 — 01d 수렴 정리에 '비주기'가 필요한 이유.
5. 예제 사슬: $\rho_{22}=0.55$, $E_2V_2=20/9$, $E_0T_0=8/3$, 흡수확률 $4/9$ vs $5/9$ — 모두 시뮬레이션과 4 SE 안에서 일치했다.

**trap 카드**
- Q: '상태 $x$의 주기가 2'라는 말은 정확히 2걸음마다 $x$로 돌아온다는 뜻인가?
- A: 아니다. $\{n:p^n(x,x)>0\}$의 최대공약수가 2라는 뜻이다. 예컨대 귀환 길이가 $\{4,6,8,\dots\}$여도 주기는 2다. 반대로 길이 3과 5가 모두 가능하면 자기 루프가 없어도 주기 1이다 (E3b).

**다음 노트북: 01c** — 일시 상태에서 출발하면 **어느** 닫힌 클래스에, **얼마 만에** 흡수되는가? 정준형의 $Q$ 블록에서 기본행렬 $N=(I-Q)^{-1}$을 만들면 두 질문이 행렬 곱 하나($B=NR$, $t=N\mathbf 1$)로 풀린다 — 오늘 검산에 슬쩍 쓴 `absorption`이 그것이고, 도박꾼 파산의 두 번째 풀이가 된다.

log/progress.md 한 줄 템플릿:
`- YYYY-MM-DD 01b 상태의 분류 | 소요 __분 | 예측 적중 _/7 | 막힌 곳: ___ | 다음: 01c`